# 10. Final Recommendation Product Rows

This notebook is the product-facing layer of the project. Earlier notebooks build the catalog, audit the data, create representations, segment the catalog, evaluate rankers, and build graph signals. This notebook asks a different question: what should the user actually see?

The output is a streaming-style set of rows rather than a single ranked list. Each row has a different job: general recommendations, similar-anime anchors, franchise continuation, people-based discovery, and controlled exploration.
The model comparison cells deliberately keep only three rankers: the best classical/product hybrid and the two best advanced learned rerankers from the latest offline evaluation. The final row output then uses one chosen ranker policy plus row-specific generators for similarity, continuation, people/staff/studios, and exploration.


## Product Profile Bands

The recommender separates true cold start from users with history.

- **Cold Starter:** no usable list yet; ask for favorite genres/tags and recognizable popular titles.
- **Beginner:** small watch history; keep rows safe, popular, short, and recognizable.
- **Casual:** enough history for similar anime and direct relations.
- **Fan:** stronger profile; mix collaborative/content taste with current discovery and people/staff signals.
- **Veteran:** many obvious titles are already known; graph, staff/VA paths, novelty, and long-tail coverage matter more.

The row mix can change by band. For example, a Beginner may not need a `give_it_a_try` row yet, while a Veteran benefits from it.

In [1]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
from IPython.display import display

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SCRIPT = ROOT / 'src' / '10_build_recommendation_product_rows.py'
ARTIFACT_DIR = ROOT / 'artifacts' / 'recommendation'
ROWS_CSV = ARTIFACT_DIR / 'product_recommendation_rows.csv'
SUMMARY_JSON = ARTIFACT_DIR / 'product_recommendation_summary.json'
METRICS_CSV = ARTIFACT_DIR / 'evaluation_metrics.csv'
ADVANCED_METRICS_CSV = ARTIFACT_DIR / 'advanced_ranker_metrics.csv'
USER_EVAL_CSV = ARTIFACT_DIR / 'user_level_eval_sample.csv'
ADVANCED_EVAL_CSV = ARTIFACT_DIR / 'advanced_ranker_eval_rows.csv'
CATALOG_CSV = ROOT / 'data' / 'processed' / 'anime_dataset.csv'

# Demo input.
# SOURCE='xml' reads the MAL export at data/raw/MyList.xml.
# In XML mode, the optional MAL username is used only for Jikan favorites.
# SOURCE='anilist' fetches both the list and favorites from AniList GraphQL.
SOURCE = (input('Profile source [xml/anilist] (default: xml): ').strip().lower() or 'xml')
if SOURCE not in {'xml', 'anilist'}:
    raise ValueError("SOURCE must be 'xml' or 'anilist'.")

if SOURCE == 'xml':
    USERNAME = input('Optional MAL username for favorites via Jikan (default: Champux, - to skip): ').strip() or 'Champux'
    if USERNAME == '-':
        USERNAME = ''
else:
    USERNAME = input('AniList username: ').strip()
    if not USERNAME:
        raise ValueError('AniList source requires a username.')

print(ROOT)
print(f"Demo profile source: {SOURCE}; username={USERNAME or 'none'}")


c:\Users\CHAMPUX\Downloads\UPC TRABAJOS 2026\BIG DATA\proyect
Demo profile source: xml; username=Champux


## Generate Rows

For MAL profiles, the notebook uses the exported XML file at `data/raw/MyList.xml` as the list source. This is intentionally preferred over the public MAL API list because the API comparison missed some scored entries from the export.

The optional MAL username is kept only to fetch favorites from Jikan `/users/{username}/favorites`, which gives favorite anime and people signals for the `people_you_like` row. AniList mode still uses a username because AniList GraphQL can fetch both list entries and favorites.

The saved output contains recommendation rows and anonymous summary counts, not the raw username list.

In [2]:
command = [sys.executable, str(SCRIPT), '--source', SOURCE]
if USERNAME:
    command += ['--username', USERNAME]

print('Running:', ' '.join(command))
result = subprocess.run(command, cwd=ROOT, text=True, capture_output=True)
print(result.stdout)
if result.returncode:
    print(result.stderr)
    raise RuntimeError(f'Product row builder failed: {result.returncode}')


Running: c:\Users\CHAMPUX\AppData\Local\Programs\Python\Python313\python.exe c:\Users\CHAMPUX\Downloads\UPC TRABAJOS 2026\BIG DATA\proyect\src\10_build_recommendation_product_rows.py --source xml --username Champux
Could not fetch Jikan favorites for MAL username Champux: Request failed for https://api.jikan.moe/v4/users/Champux/favorites: None
{
  "profile_band": "Veteran",
  "profile_source": "mylist_xml_with_mal_favorites",
  "username": "Champux",
  "scored_count": 1022,
  "positive_count": 811,
  "known_count": 1339,
  "favorite_anime_ids_used": 0,
  "favorite_voice_actor_ids_used": 0,
  "favorite_staff_ids_used": 0,
  "favorite_people_names_used": 0,
  "favorite_studio_names_used": 0,
  "hentai_majority": false,
  "general_recommendation_source": "balanced_current_popular_content",
  "selected_recommender_models": {
    "classical_backbone": {
      "method": "level_tuned_product_hybrid",
      "balanced_profile_hit_at_12": 0.941081,
      "hit_rate_at_12": 0.955169,
      "ndcg_

## Product Summary

In [3]:
summary = json.loads(SUMMARY_JSON.read_text(encoding='utf-8'))
summary


{'profile_band': 'Veteran',
 'profile_source': 'mylist_xml_with_mal_favorites',
 'username': 'Champux',
 'scored_count': 1022,
 'positive_count': 811,
 'known_count': 1339,
 'favorite_anime_ids_used': 0,
 'favorite_voice_actor_ids_used': 0,
 'favorite_staff_ids_used': 0,
 'favorite_people_names_used': 0,
 'favorite_studio_names_used': 0,
 'hentai_majority': False,
 'general_recommendation_source': 'balanced_current_popular_content',
 'selected_recommender_models': {'classical_backbone': {'method': 'level_tuned_product_hybrid',
   'balanced_profile_hit_at_12': 0.941081,
   'hit_rate_at_12': 0.955169,
   'ndcg_at_12': 0.746538,
   'map_at_12': 0.680129},
  'advanced_candidates': [{'method': 'xgboost_pairwise_ranker',
    'balanced_profile_hit_at_12': 0.950791,
    'hit_rate_at_12': 0.961674,
    'ndcg_at_12': 0.760622,
    'map_at_12': 0.69655},
   {'method': 'torch_feature_mlp_reranker',
    'balanced_profile_hit_at_12': 0.950702,
    'hit_rate_at_12': 0.961359,
    'ndcg_at_12': 0.7603

## Top Evaluated Models

The UI is row-based, so no single model owns the whole page. Still, the top evaluated models tell us which ranker should power the general row and which ones are safer as fallbacks. This table compares the top five classical/product and advanced models using the same `@12` product-row metrics.


In [4]:
metrics = pd.read_csv(METRICS_CSV) if METRICS_CSV.exists() else pd.DataFrame()
advanced_metrics = pd.read_csv(ADVANCED_METRICS_CSV) if ADVANCED_METRICS_CSV.exists() else pd.DataFrame()
rank_cols = ['balanced_profile_hit_at_12', 'hit_rate_at_12', 'ndcg_at_12', 'map_at_12']

if metrics.empty:
    print('Run notebook 08 first so evaluation metrics exist.')
    selected_models = pd.DataFrame()
else:
    classical = metrics.copy()
    if 'model_layer' in classical.columns:
        classical = classical[classical['model_layer'].fillna('').str.contains('classical|product', case=False, regex=True)]
    best_classical = classical.sort_values([c for c in rank_cols if c in classical.columns], ascending=False).head(1)
    best_classical = best_classical.assign(selection_role='best_classical_product')

    if advanced_metrics.empty:
        best_advanced = pd.DataFrame(columns=best_classical.columns)
    else:
        best_advanced = advanced_metrics.sort_values([c for c in rank_cols if c in advanced_metrics.columns], ascending=False).head(2)
        roles = ['advanced_candidate_1', 'advanced_candidate_2'][:len(best_advanced)]
        best_advanced = best_advanced.assign(model_layer='advanced_learned', selection_role=roles)

    selected_models = pd.concat([best_classical, best_advanced], ignore_index=True, sort=False)
    selected_cols = ['selection_role', 'model_layer', 'method', 'hit_rate_at_12', 'ndcg_at_12', 'map_at_12', 'balanced_profile_hit_at_12']
    display(selected_models[[c for c in selected_cols if c in selected_models.columns]])


,selection_role,model_layer,method,hit_rate_at_12,ndcg_at_12,map_at_12,balanced_profile_hit_at_12
0,best_classical_product,NaN,level_tuned_product_hybrid,0.955169,0.746538,0.680129,0.941081
1,advanced_candidate_1,advanced_learned,xgboost_pairwise_ranker,0.961674,0.760622,0.696550,0.950791
2,advanced_candidate_2,advanced_learned,torch_feature_mlp_reranker,0.961359,0.760316,0.696227,0.950702


## Offline Row Comparison for Top Models

The product rows below are generated for the demo profile. This extra check compares the top evaluated models on a real held-out evaluation user, showing the 12 titles each model would place in the visible row. It is not the final UI output, but it helps explain why the general row should use the strongest evaluated ranker while specialist rows still use graph, people, and exploration logic.


In [5]:
eval_rows_path = ARTIFACT_DIR / 'advanced_ranker_eval_rows.csv'
classical_rows_path = ARTIFACT_DIR / 'user_level_eval_sample.csv'

if 'selected_models' not in globals() or selected_models.empty:
    print('Select models first.')
elif not eval_rows_path.exists() and not classical_rows_path.exists():
    print('No model-level eval rows available yet. Run notebook 08 first.')
else:
    model_methods = selected_models['method'].dropna().astype(str).tolist()
    frames = []
    if classical_rows_path.exists():
        classical_rows = pd.read_csv(classical_rows_path)
        frames.append(classical_rows[classical_rows['method'].isin(model_methods)])
    if eval_rows_path.exists():
        advanced_rows = pd.read_csv(eval_rows_path)
        frames.append(advanced_rows[advanced_rows['method'].isin(model_methods)])
    compare_rows = pd.concat(frames, ignore_index=True, sort=False) if frames else pd.DataFrame()
    if compare_rows.empty:
        print('Selected models were not found in available row-level artifacts.')
    else:
        counts = compare_rows.groupby('userID')['method'].nunique().sort_values(ascending=False)
        sample_user = int(counts.index[0])
        sample = compare_rows[compare_rows['userID'].eq(sample_user)].copy()
        print(f'Raw top-12 comparison userID={sample_user}; models={model_methods}')
        raw_rows = []
        for method in model_methods:
            part = sample[sample['method'].eq(method)].sort_values('rank').head(12).copy()
            title_col = 'title' if 'title' in part.columns else 'candidate_title'
            if title_col not in part.columns:
                title_col = 'holdout_title'
            for rank, row in enumerate(part.itertuples(index=False), start=1):
                raw_rows.append({
                    'method': method,
                    'rank': rank,
                    'mal_id': getattr(row, 'mal_id', getattr(row, 'candidate_mal_id', None)),
                    'title': getattr(row, title_col, ''),
                    'score': getattr(row, 'score', getattr(row, 'model_score', None)),
                })
        raw_model_rows = pd.DataFrame(raw_rows)
        display(raw_model_rows)
        chosen_model = model_methods[1] if len(model_methods) > 1 else model_methods[0]
        print(f'Chosen main-row policy for the demo: {chosen_model}. Row-specific generators still handle continuation, similarity anchors, people/studios, and exploration.')


Raw top-12 comparison userID=2147334841; models=['level_tuned_product_hybrid', 'xgboost_pairwise_ranker', 'torch_feature_mlp_reranker']


,method,rank,mal_id,title,score
0,level_tuned_product_hybrid,1,None,,None
1,xgboost_pairwise_ranker,1,None,,None
2,torch_feature_mlp_reranker,1,None,,None


Chosen main-row policy for the demo: xgboost_pairwise_ranker. Row-specific generators still handle continuation, similarity anchors, people/studios, and exploration.


## Row Output

Each row is capped at 12 titles. The `reason` field is intentionally user-facing enough to become UI text later, while `anchor` tells us which title/person caused the recommendation when applicable.

In [6]:
rows = pd.read_csv(ROWS_CSV)
display(rows[['row', 'rank', 'title', 'score', 'reason', 'anchor', 'mal_url']].head(80))


,row,rank,title,score,reason,anchor,mal_url
0,general_recommendations,1,Tongari Boushi no Atelier,0.6410,Summer 2026 window: current/recent show with p...,current season window,https://myanimelist.net/anime/51553/Tongari_Bo...
1,general_recommendations,2,Ikoku Nikki,0.6379,Summer 2026 window: current/recent show with p...,current season window,https://myanimelist.net/anime/58788/Ikoku_Nikki
2,general_recommendations,3,Super no Ura de Yani Suu Futari,0.6210,Summer 2026 window: current/recent show with p...,current season window,https://myanimelist.net/anime/62076/Super_no_U...
3,general_recommendations,4,Kimi no Suizou wo Tabetai,0.8207,popular high-score anchor with mild profile fit,popular high-score,https://myanimelist.net/anime/36098/Kimi_no_Su...
4,general_recommendations,5,Rainbow: Nisha Rokubou no Shichinin,0.8057,popular high-score anchor with mild profile fit,popular high-score,https://myanimelist.net/anime/6114/Rainbow__Ni...
5,general_recommendations,6,Josee to Tora to Sakana-tachi,0.8051,popular high-score anchor with mild profile fit,popular high-score,https://myanimelist.net/anime/40787/Josee_to_T...
6,general_recommendations,7,Aka no Kioku,0.5656,content fallback: profile match + catalog quality,content fallback,https://myanimelist.net/anime/61332/Aka_no_Kioku
7,general_recommendations,8,"Let's Roll, Cinnamoroll!",0.5647,content fallback: profile match + catalog quality,content fallback,https://myanimelist.net/anime/61151/Lets_Roll_...
8,general_recommendations,9,Noto Hantou Fukkou Ouen Kikaku,0.5583,content fallback: profile match + catalog quality,content fallback,https://myanimelist.net/anime/63222/Noto_Hanto...
9,general_recommendations,10,Mofusand,0.5566,content fallback: profile match + catalog quality,content fallback,https://myanimelist.net/anime/63151/Mofusand


## Filter Layer

Filters should apply after row generation so the system can keep the recommendation reason while respecting user control. The intended filters are: year range, genres/tags, content rating, hentai toggle, runtime, episode count, and whether to hide not-yet-finished airing shows.

Hentai is off by default unless the profile is clearly majority hentai. This is treated as a product safety/default setting, not as a judgment about the user.

## Next Product Work

The row builder is now ready for a small UI prototype: ask MAL users to upload their exported XML file, optionally ask for a MAL username to fetch favorites, or ask AniList users for their username. Then classify the user profile band, generate recommendation rows, let the user apply filters, reroll individual rows, and export selected titles with MAL/AniList links.

For MAL profiles, the list comes from XML and favorite anime/voice actors can be read from Jikan favorites. For AniList profiles, favorite anime, staff, and studios can be read through GraphQL. The `people_you_like` row now uses VAs, directors/creators, and studios.